# Sequence profile algorithm

In [1]:
###imports
import numpy as np
import os

cwd = os.getcwd()
parent = os.path.dirname(cwd)
grandparent = os.path.dirname(parent)

data_dir =  f"{parent}/data/"

In [15]:
###load protein sequences 
seqs = []
#location = []
encoding = {}

alphabet_file = data_dir + "alphabet"
alphabet = np.loadtxt(alphabet_file, dtype=str)

for i, AA in enumerate(alphabet):
    encoding[AA] = i

with open(data_dir + "signalpeptide.csv") as f:
    #print(f.read().split())
    for i,line in enumerate(f.read().split()):
        #dont include the first line
        if i > 1:
            
            current_sequence = line.split(",")[0]
            enc_sequence = []
            for AA in current_sequence:
                enc_sequence.append( encoding[AA] )
                #encode the sequence to be between 1 and 20 (corresponding to an AA)
            seqs.append(enc_sequence)

#output:
## two lists, 
### one with 30-mer peptides
### one with translocation

In [16]:
def Initialize_model(seqs):
    ### Initialize architecture
    sequence_length = len(seqs[0])

    #sequence always starts with M, so initial propability for M is 1, and everything else is 0
    initial_prop = [0 if p != "M" else 1 for p in alphabet ]
    #print(initial_prop)

    #####states
    states = [f"P{1}", "I1"]

    for i in range(1, sequence_length-2):
        states.append(f"P{i+1}")
        states.append(f"I{i+1}")
        states.append(f"D{i+1}")

    #make last 2 states, which are gonna be unique
    states.append(f"P{sequence_length-1}")
    states.append(f"I{sequence_length-1}")
    states.append(f"P{sequence_length}")

    #all states can go to end state
    states.append("end")

    ### Emissions (each row is a state and each column is an emission from the alphabet)
    #initial emission propabilty (equal propabilty for all AA)
    p_initial_emission = [1/len(alphabet)]*len(alphabet) + [0]
    d_emission = [0] *len(alphabet) + [1] # always ""
    i_initial_emission = [1/len(alphabet)]*len(alphabet) + [0]

    emission = np.array([
        p_initial_emission if state.startswith("P")
        else d_emission if state.startswith("D")
        else i_initial_emission if state.startswith("I")
        else d_emission  # end state
        for state in states
    ])


    ###initial transmission propability
    transmission = np.zeros ( (len(states), len(states)) )

    for i, s in enumerate(states):
        #if it is not the last state
        if i != len(states) -1:
            if s[0] == "P":
                #check if there exists an I state
                if states[i+1].startswith("I"):
                    transmission[i,i+1] = 1
                    #is it followed by a D state
                    if states[i+2].startswith("D"):
                        transmission[i,i+2] = 1
                        #is it followed by a P state
                        if states[i+3].startswith("P"):
                            transmission[i,i+3] = 1
                    elif states[i+2].startswith("P"):
                            transmission[i,i+2] = 1
            elif s[0] == "I":
                transmission[i,i] = 1
                #check if there exists an P state next
                if states[i+1].startswith("P"):
                    transmission[i,i+1] = 1
                elif states[i+2].startswith("P"):
                    transmission[i,i+1] = 1
            elif s[0] == "D":

                if states[i+2].startswith("P"):
                    transmission[i,i+2] = 1
                    if i < len(states) - 2:
                        if states[i+3].startswith("D"):
                            transmission[i,i+3] = 1
        else:
            transmission[i,i] = 1

    initial_transmission = transmission

    return initial_prop, transmission, emission, initial_transmission, sequence_length, states

### Functions

In [17]:

def calculate_alpha(input_encode, states, transmission, emission, initial_transmission, weight = 1):
    def initialize_forward(input_encode, states):
        alpha = np.zeros(shape=(len(states), len(input_encode)))
        alpha[0][0] = 1
        return alpha
    
    #make first row (shape = (states, sequence length))
    alpha = initialize_forward(input_encode, states)

    # main loop
    #for each position in sequence
    for i in range(1, len(input_encode)):
        #for each state, j at new position i
        for j in range(0, len(states)):

            _sum = 0
            #sum over all states, k of old position i
            for k in range(0, len(states)):
                if initial_transmission[k,j] != 0:
                    if states[j].startswith("D"):
                        #Deletion state does not move forwards in the sequence and has no emision
                        _sum += alpha[k][i] * transmission[k][j]
                    else:
                        _sum += alpha[k][i-1]*transmission[k][j]*emission[j][input_encode[i]]
            
            # store prob
            alpha[j][i] = _sum * weight
    return alpha

def calculate_beta(input_encode, states, transmission, emission, initial_transmission,weight = 1):

    def initialize_backward(input_encode, states):
        beta = np.zeros(shape=(len(states), len(input_encode)))
        for i in range(0, len(states)):
            beta[i][-1] = 1
        return beta

    beta = initialize_backward(input_encode, states)


    # main loop
    # for each position in sequence starting from second to last
    for i in range(len(input_encode)-2, -1, -1):
        #for each state in current j
        for j in range(0, len(states)):

            _sum = 0
            # for each state in the next position (i+1)
            for k in range(0, len(states)):
                #if transmission is possible
                if initial_transmission[j,k] != 0:
                    _sum += emission[k][input_encode[i+1]] * beta[k][i+1] *transmission[j][k]
            
            # store prob
            beta[j][i] = _sum * weight

    return beta

def calculate_gamma(input_encode, states, alpha, beta):
    gamma = np.zeros(shape=(len(states), len(input_encode)))

    #for each position t
    for t in range(len(input_encode)):
        #calculate alpha[i]*beta[i] for the entire column
        for i in range(len(states)):
            gamma[i][t] = alpha[i][t] * beta[i][t] #not normalized yet!
            #print(gamma[i][t])
        #normalize entire column by the sum of the column
        #print(gamma[:][t])
        c_sum = sum(gamma[:,t])
        for i in range(len(states)):
            if c_sum != 0:
                gamma[i][t] = gamma[i][t] / c_sum
    return gamma

def calculate_xi(input_encode, states, alpha, beta, initial_transmission, sequence_length, transmission, emission):
    #Initialize xi (state from, state to) (we will sum over all positions (not last position) to imidiatly calculate the sum used in a_new)
    xi = np.zeros(shape= (len(states), len(states), sequence_length))

    #for each from state
    for i in range(len(states)):
        #for each to state
        for j in range(len(states)):
            #if transition from i to j is possible (not 0 at the beginning)
            if initial_transmission[i,j] != 0:
                #_sum = 0
                for t in range(sequence_length-1):
                    xi[i,j,t]= alpha[i][t] * transmission[i,j]* emission[j,input_encode[t+1]] * beta[j,t+1]
                #xi_psum[i,j] = _sum/aT_sum

    #renormalize
    #xi[:,:,t] /= np.sum(xi[:,:,t])
    denom = np.sum(xi[:,:,t])
    if denom > 0:
        xi[:,:,t] /= denom
    ## returns xi summed over all t
    return np.sum(xi, axis=2)

def calculate_new_transmission(gamma, xi_psum, initial_transmission, states):
    
    gamma_sum = np.zeros(shape = (len(states)))
    for i in range(len(states)):
        gamma_sum[i] = np.sum(gamma[i,:-1])
        
    trans_new = initial_transmission.copy()
    for i in range(len(states)):
        for j in range(len(states)):
            if initial_transmission[i,j] != 0:
                #include max to avoid division by 0

                #For multiple sequences is is the sum of xi_psum for all sequences divided by the sum of gamma_sum[i] for all sequences 
                trans_new[i,j] = xi_psum[i,j]/max(gamma_sum[i],0.00000001)
                
    for i in range(len(states)):
        row_sum = np.sum(trans_new[i,:])
        if row_sum > 0:
            trans_new[i,:] /= row_sum
    return trans_new

def calculate_new_emission(input_encode, states, gamma, emission, sequence_length):
    #the positions of the states that are not D and end
    pi_states = [i for i,s in enumerate(states) if s.startswith(("P","I")) ]
    #print(alphabet)
    emis_new = emission.copy()

    #for state i that is eiter P or I state (don't update deletions and end)
    for i in pi_states:
        for a in alphabet:
            _sum = 0
            #sum of all observations that are equal to current symbol
            for t in range(sequence_length):
                if input_encode[t] == encoding[a]:
                    _sum += gamma[i,t]
            
            emis_new[i,encoding[a]] = _sum / max(np.sum(gamma[i,:]), 0.0001)
    return emis_new


## Main Loop

In [29]:
initial_prop, transmission, emission, initial_transmission, sequence_length, states = Initialize_model(seqs)
iterations = 5

#split data into chunks
chunk_size = 5
n_chunks = len(seqs)//chunk_size

chunk_count = 0


for i in range(iterations):
    print(f"iteration {i}")
    #iterate over all sequences
    xi_seq_sum = np.zeros(shape= (len(states), len(states)))
    gamma_seq_sum = np.zeros(shape=(len(states), sequence_length))

    #calculate current chunk
    c_start = (chunk_count % n_chunks) * chunk_size
    c_end = min(c_start+chunk_size, len(seqs))
    
    for s in seqs[c_start: c_end]:
        input = s
        #print("calculate alpha")
        alpha = calculate_alpha(input, states, transmission, emission, initial_transmission)
        #print("b")
        beta =calculate_beta(input, states, transmission, emission, initial_transmission)
        gamma = calculate_gamma(input, states, alpha, beta)
        xi_psum = calculate_xi(input, states, alpha, beta, initial_transmission, sequence_length, transmission, emission)
        xi_seq_sum += xi_psum
        gamma_seq_sum += gamma

    
    transmission = calculate_new_transmission(gamma_seq_sum, xi_seq_sum, initial_transmission, states)
    #print(np.sum(transmission, axis=1)[0])
    emission = calculate_new_emission(input, states, gamma_seq_sum, emission, sequence_length)
    


222
iteration 0
iteration 1
iteration 2


KeyboardInterrupt: 